In [21]:
import pandas as pd
import numpy as np
import os
from scipy import stats
import matplotlib.pyplot as plt

In [22]:
def t_test(non_mace_avg, non_mace_std, non_mace_group_size, mace_avg, mace_std, mace_group_size): # std 樣本標準差 (N-1)
    t_statistic, p_value = stats.ttest_ind_from_stats(mean1=non_mace_avg, std1=non_mace_std, nobs1=non_mace_group_size, mean2=mace_avg, std2=mace_std, nobs2=mace_group_size)
    if p_value is not np.float32:
        p_value = np.float32(p_value)
    return t_statistic, p_value

In [23]:
non_mace_dir = r'V:\dunwei\MACE\dataset\ECG_Founder_result\sr500_0.5_50_10s_ECG_signal_rpeak_5_10min_longer\sr500_0.5_50_10s_ECG_signal_feature_std/' ###
mace_dir = r'V:\dunwei\MACE\dataset\ECG_Founder_result\sr500_0.5_50_10s_ECG_signal_rpeak_5_10min_longer\sr500_0.5_50_10s_ECG_signal_feature_std/' ###
save_dir = r'V:\dunwei\MACE\dataset\ECG_Founder_result\sr500_0.5_50_10s_ECG_signal_rpeak_5_10min_longer\sr500_0.5_50_10s_ECG_signal_feature_std/' ###

In [24]:
non_mace_raw_data = pd.read_csv(os.path.join(non_mace_dir, 'non_mace_feature_category_std.csv')) ###
mace_raw_data = pd.read_csv(os.path.join(mace_dir, 'mace_feature_category_std.csv')) ###
print(non_mace_raw_data.shape, mace_raw_data.shape)

(432, 151) (90, 151)


In [25]:
p_value_list = [] 

if non_mace_raw_data.shape[1] == mace_raw_data.shape[1]:
    non_mace_data = non_mace_raw_data.drop(columns=['research_id'])
    mace_data = mace_raw_data.drop(columns=['research_id'])

    non_mace_data_avg = np.float32(np.mean(non_mace_data, axis=0))
    non_mace_data_std = np.float32(np.std(non_mace_data, axis=0, ddof=1)) # 樣本標準差

    mace_data_avg = np.float32(np.mean(mace_data, axis=0))
    mace_data_std = np.float32(np.std(mace_data, axis=0, ddof=1)) # 樣本標準差

    non_mace_avg_std = np.vstack((non_mace_data_avg, non_mace_data_std))
    mace_avg_std = np.vstack((mace_data_avg, mace_data_std))

    for i in range(non_mace_avg_std.shape[1]):
        _, p_value = t_test(non_mace_avg_std[0,i], non_mace_avg_std[1,i], non_mace_data.shape[0], mace_avg_std[0,i], mace_avg_std[1,i], mace_data.shape[0])
        p_value_list.append((p_value))  


In [26]:
non_mace_avg_std_df = pd.DataFrame(non_mace_avg_std, columns=[int(idx) for idx in range(1,(non_mace_avg_std.shape[1]+1),1)])
mace_avg_std_df = pd.DataFrame(mace_avg_std, columns=[int(idx) for idx in range(1,(non_mace_avg_std.shape[1]+1),1)])
p_value_results_df = pd.DataFrame(p_value_list, index=[int(idx) for idx in range(1,(non_mace_avg_std.shape[1]+1),1)]).T
tmp_result_df = pd.concat([non_mace_avg_std_df, mace_avg_std_df, p_value_results_df], axis=0, ignore_index=True)

group_name = ['non_mace_avg', 'non_mace_std', 'mace_avg', 'mace_std', 'p_value']
group_name_df = pd.DataFrame(group_name, columns=['group_statistics'])
result_df = pd.concat([group_name_df, tmp_result_df], axis=1)

result_df.to_csv(os.path.join(save_dir, 'category_ttest.csv'), index=False)

In [27]:
draw_points = 15
for i in range(len(p_value_list)//15):
    start_idx = i*draw_points
    end_idx = (i+1)*draw_points
    x_index = range(start_idx+1, end_idx+1)
    plt.figure(figsize=(10, 6))
    plt.bar(x_index, p_value_list[start_idx:end_idx], color='tab:blue', width=0.6, label='p value')
    plt.xticks(x_index)
    plt.xlabel('Category Index')
    plt.ylabel('p-value')
    plt.title(f'Category p-values from {start_idx+1} to {end_idx}')
    plt.axhline(y=0.05, color='r', linestyle='--', label='p value = 0.05')
    plt.legend()
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'category_p_value_{start_idx+1}_to_{end_idx}.png'))
    plt.close()


In [28]:
category_significant = []
for i in range(len(p_value_list)):
    if p_value_list[i] < 0.05:
        category_significant.append(i+1)
        print(f'Category {i+1}: p-value = {p_value_list[i]}')
print(category_significant)

Category 26: p-value = 0.032200515270233154
Category 33: p-value = 0.032224513590335846
Category 56: p-value = 0.03278769180178642
Category 59: p-value = 0.001279624062590301
Category 61: p-value = 0.018919089809060097
Category 63: p-value = 0.04145844280719757
Category 90: p-value = 0.0474766306579113
Category 137: p-value = 0.02056923508644104
Category 138: p-value = 0.012778903357684612
Category 140: p-value = 0.008772704750299454
Category 142: p-value = 0.03848732262849808
[26, 33, 56, 59, 61, 63, 90, 137, 138, 140, 142]


In [29]:
with open('tasks.txt','r') as file:
    all_significant_feature = [line.strip() for line in file.readlines()]

In [30]:
significant_feature = [f'{all_significant_feature[i-1]} ({i})' for i in category_significant]
non_mace_mace_df = result_df.copy()
tmp_significant_feature_df = non_mace_mace_df[category_significant].copy()
tmp_significant_feature_df.columns = significant_feature
significant_feature_df = pd.concat([non_mace_mace_df['group_statistics'], tmp_significant_feature_df], axis=1)
display(significant_feature_df)
significant_feature_df.to_csv(os.path.join(save_dir, 'significant_feature_ttest.csv'), index=False)

,group_statistics,NONSPECIFIC ST ABNORMALITY (26),ATRIAL FLUTTER (33),WITH REPOLARIZATION ABNORMALITY (56),WIDE QRS RHYTHM (59),RIGHT ATRIAL ENLARGEMENT (61),INCOMPLETE LEFT BUNDLE BRANCH BLOCK (63),NONSPECIFIC INTRAVENTRICULAR CONDUCTION DELAY (90),SINUS/ATRIAL CAPTURE (137),AV DUAL-PACED COMPLEXES (138),RBBB AND LEFT POSTERIOR FASCICULAR BLOCK (140),ATRIAL-PACED COMPLEXES (142)
0,non_mace_avg,0.059406,0.059181,0.066862,0.053812,0.059265,0.065889,0.063311,0.066755,0.058706,0.063500,0.053959
1,non_mace_std,0.019022,0.019100,0.020172,0.016963,0.019424,0.021815,0.019863,0.022504,0.018309,0.022069,0.014065
2,mace_avg,0.054749,0.054515,0.061925,0.047508,0.054121,0.060834,0.058794,0.060942,0.053621,0.056885,0.050596
3,mace_std,0.017135,0.016969,0.018564,0.015997,0.015812,0.018901,0.018430,0.016508,0.013385,0.019813,0.013620
4,p_value,0.032201,0.032225,0.032788,0.001280,0.018919,0.041458,0.047477,0.020569,0.012779,0.008773,0.038487
